# Synthetic Food Waste Forecasting Dataset
Mimics the Kaggle original (`food_waste_university_canteen.csv`) by fitting statistical priors from the real data and generating a 6-month half-hourly synthetic dataset.

## 1. Imports

In [21]:
import numpy as np
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar
from scipy.stats import gamma

## 2. Load original Kaggle data

In [22]:
original = pd.read_csv("data/original/food_waste_university_canteen.csv")
original['Date'] = pd.to_datetime(original['Date'])
original['day_of_week'] = original['Date'].dt.dayofweek

print(f"Loaded {len(original):,} rows")
print(original.head())

Loaded 2,600 rows
        Date       Meal Canteen_Section Food_Category  Waste_Weight_kg  \
0 2025-07-30     Dinner               B    Vegetables             1.50   
1 2025-06-15  Breakfast               B          Rice             3.69   
2 2025-07-29  Breakfast               A          Soup             1.54   
3 2025-07-17  Breakfast               A          Soup             1.81   
4 2025-07-03     Dinner               D          Rice             4.69   

   Unit_Price_per_kg  Cost_Loss  day_of_week  
0                3.0       4.50            2  
1                2.0       7.38            6  
2                1.5       2.31            1  
3                1.5       2.71            3  
4                2.0       9.38            3  


## 3. Extract statistical priors from real data

In [23]:
waste_positive = original[original['Waste_Weight_kg'] > 0]
overall_mean = waste_positive['Waste_Weight_kg'].mean()
overall_std  = waste_positive['Waste_Weight_kg'].std()

# Per-meal mean / std
meal_stats = {}
for meal in ['Breakfast', 'Lunch', 'Dinner']:
    data = waste_positive[waste_positive['Meal'] == meal]['Waste_Weight_kg']
    meal_stats[meal] = {
        'mean': data.mean() if len(data) > 5 else overall_mean,
        'std':  data.std()  if len(data) > 5 else overall_std,
    }

# Location and day-of-week multipliers
location_multiplier = (
    waste_positive.groupby('Canteen_Section')['Waste_Weight_kg'].mean() / overall_mean
).to_dict()

dow_multiplier = (
    waste_positive.groupby('day_of_week')['Waste_Weight_kg'].mean() / overall_mean
).to_dict()

print(f"Overall waste: mean={overall_mean:.2f}, std={overall_std:.2f}")
print(f"Location multipliers: {location_multiplier}")
print(f"Day-of-week multipliers: {dow_multiplier}")

Overall waste: mean=2.59, std=1.41
Location multipliers: {'A': 0.9720387461047092, 'B': 1.0108876237334945, 'C': 1.0224485866827961, 'D': 0.9947661182421882}
Day-of-week multipliers: {0: 0.9742805501037202, 1: 0.9442401441321133, 2: 0.9857975979535906, 3: 1.0425593739981225, 4: 0.9755101123831222, 5: 1.0207426237487853, 6: 1.046505056093712}


## 4. Build continuous half-hourly timeline

In [24]:
RNG   = np.random.default_rng(seed=2025)
START = "2025-01-01 00:00"
END   = "2025-06-30 23:30"

idx = pd.date_range(start=START, end=END, freq="30min")
n   = len(idx)
print(f"Timeline: {idx[0]} → {idx[-1]}  ({n:,} rows, {n/48:.0f} days)")

df = pd.DataFrame({"timestamp": idx})
df["date"]        = df["timestamp"].dt.date
df["year"]        = df["timestamp"].dt.year
df["month"]       = df["timestamp"].dt.month
df["day"]         = df["timestamp"].dt.day
df["hour"]        = df["timestamp"].dt.hour
df["minute"]      = df["timestamp"].dt.minute
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["week_of_year"]= df["timestamp"].dt.isocalendar().week.astype(int)
df["is_weekend"]  = (df["day_of_week"] >= 5).astype(int)
df["hour_frac"]   = df["hour"] + df["minute"] / 60.0
df["day_index"]   = (df["timestamp"] - df["timestamp"].iloc[0]).dt.days

Timeline: 2025-01-01 00:00:00 → 2025-06-30 23:30:00  (8,688 rows, 181 days)


## 5. Add cyclical time features

In [25]:
df["hour_sin"]  = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"]  = np.cos(2 * np.pi * df["hour"] / 24)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

print("Cyclical features added.")

Cyclical features added.


## 6. Assign meal type and location

In [26]:
def assign_meal(hour):
    if   6  <= hour < 10: return 'Breakfast'
    elif 11 <= hour < 14: return 'Lunch'
    elif 17 <= hour < 21: return 'Dinner'
    else:                 return 'Closed'

df['meal'] = df['hour'].apply(assign_meal)

locations = list(location_multiplier.keys()) or ['A', 'B', 'C', 'D']
df['location_id'] = RNG.choice(locations, size=n)

print(df['meal'].value_counts())

meal
Closed       4706
Breakfast    1448
Dinner       1448
Lunch        1086
Name: count, dtype: int64


## 7. Generate waste_kg using Gamma distribution (vectorized)

In [27]:
trend = 1.0 + 0.0012 * df["day_index"].values

# Map meal / location / dow factors onto every row
base_mean_arr = df['meal'].map({m: s['mean'] for m, s in meal_stats.items()}).fillna(overall_mean).values
base_cv_arr   = df['meal'].map({m: (s['std'] / s['mean'] if s['mean'] > 0 else 0.5) for m, s in meal_stats.items()}).fillna(0.5).values
dow_factor    = df['day_of_week'].map(dow_multiplier).fillna(1.0).values
loc_factor    = df['location_id'].map(location_multiplier).fillna(1.0).values

target_mean = base_mean_arr * dow_factor * loc_factor * trend
target_std  = target_mean * base_cv_arr

shape = np.clip((target_mean ** 2) / np.where(target_std > 0, target_std ** 2, 1), 0.5, 50)
scale = np.where(target_mean > 0, (target_std ** 2) / np.where(target_mean > 0, target_mean, 1), 1.0)

waste_kg = gamma.rvs(shape, scale=scale, random_state=42)
waste_kg = np.clip(np.round(waste_kg, 2), 0.05, 6.0)

# Closed hours get near-zero noise
closed_mask = (df['meal'] == 'Closed').values
waste_kg[closed_mask] = np.round(RNG.uniform(0, 0.3, closed_mask.sum()), 2)

df["waste_kg"] = waste_kg
print(f"waste_kg: mean={df['waste_kg'].mean():.2f}, std={df['waste_kg'].std():.2f}")

waste_kg: mean=1.37, std=1.63


## 8. Generate footfall correlated with waste_kg

In [28]:
def get_base_footfall(hour_frac):
    breakfast  = 18 * np.exp(-0.5 * ((hour_frac -  8.0) / 1.0) ** 2)
    lunch      = 35 * np.exp(-0.5 * ((hour_frac - 12.5) / 1.2) ** 2)
    dinner     = 30 * np.exp(-0.5 * ((hour_frac - 18.5) / 1.4) ** 2)
    late_snack =  8 * np.exp(-0.5 * ((hour_frac - 21.5) / 0.8) ** 2)
    base = (breakfast + lunch + dinner + late_snack) * 0.15
    if hour_frac < 5 or hour_frac > 23:
        base += 1.5
    return base

df['base_footfall'] = df['hour_frac'].apply(get_base_footfall)

footfall_raw = ((df["waste_kg"] - df["base_footfall"]) / 0.045) + RNG.normal(0, 0.2, n)
footfall_raw = np.clip(np.round(footfall_raw), 0, 250).astype(int)
footfall_raw[closed_mask] = RNG.integers(0, 15, size=closed_mask.sum())

df["footfall_raw"] = footfall_raw
df["footfall"]     = footfall_raw.copy()

print("Footfall generated.")

Footfall generated.


## 9. Apply holiday flags and multipliers

In [29]:
cal = USFederalHolidayCalendar()
holidays = cal.holidays(start=START, end=END)
extra_holidays = pd.to_datetime([
    "2025-01-31", "2025-02-14", "2025-02-17", "2025-03-17",
    "2025-04-18", "2025-04-20", "2025-05-05", "2025-06-15", "2025-06-19"
])
all_holidays  = holidays.union(extra_holidays)
holiday_dates = set(all_holidays.date)
df["is_holiday"] = df["date"].apply(lambda d: int(d in holiday_dates))

holiday_name_map = {
    pd.Timestamp("2025-01-01").date(): "New Year's Day",
    pd.Timestamp("2025-01-20").date(): "MLK Day",
    pd.Timestamp("2025-02-14").date(): "Valentine's Day",
    pd.Timestamp("2025-02-17").date(): "Presidents Day",
    pd.Timestamp("2025-03-17").date(): "St. Patrick's Day",
    pd.Timestamp("2025-04-18").date(): "Good Friday",
    pd.Timestamp("2025-04-20").date(): "Easter",
    pd.Timestamp("2025-05-05").date(): "Cinco de Mayo",
    pd.Timestamp("2025-05-26").date(): "Memorial Day",
    pd.Timestamp("2025-06-15").date(): "Father's Day",
    pd.Timestamp("2025-06-19").date(): "Juneteenth",
}
for hd in all_holidays:
    if hd.date() not in holiday_name_map:
        holiday_name_map[hd.date()] = "Holiday"
df["holiday_name"] = df["date"].map(holiday_name_map).fillna("")

holiday_mult_map = {
    "New Year's Day": 1.20, "Valentine's Day": 1.55, "St. Patrick's Day": 1.45,
    "Easter": 0.65,         "Good Friday": 0.80,     "Cinco de Mayo": 1.40,
    "Memorial Day": 1.30,   "Father's Day": 1.35,    "Juneteenth": 1.10,
    "MLK Day": 0.90,        "Presidents Day": 0.88,  "Holiday": 1.05,
}
df['holiday_mult'] = df['holiday_name'].map(lambda n: holiday_mult_map.get(n, 1.0))
df['waste_kg']  = (df['waste_kg']  * df['holiday_mult']).round(2)
df['footfall']  = (df['footfall']  * df['holiday_mult']).astype(int)

print(f"Holidays flagged: {df['is_holiday'].sum():,} rows")

Holidays flagged: 576 rows


## 10. Apply special event boosts

In [30]:
special_events = {
    "2025-01-12": "NFL Playoff Watch Party",
    "2025-02-02": "Super Bowl Watch Party",
    "2025-02-08": "Valentine's Dinner Promo",
    "2025-03-22": "March Madness Weekend",
    "2025-03-23": "March Madness Weekend",
    "2025-04-04": "Campus Career Fair",
    "2025-04-05": "Campus Career Fair",
    "2025-04-27": "Spring Festival",
    "2025-05-10": "Mother's Day Brunch",
    "2025-05-24": "Memorial Weekend BBQ",
    "2025-05-25": "Memorial Weekend BBQ",
    "2025-06-07": "Outdoor Concert Night",
    "2025-06-21": "Summer Solstice Bash",
    "2025-06-28": "Pride Month Event",
}
event_date_map = {pd.Timestamp(k).date(): v for k, v in special_events.items()}
df["special_event"] = df["date"].map(event_date_map).fillna("")

event_mask = (df["special_event"] != "").values
df.loc[event_mask, "waste_kg"] = (
    df.loc[event_mask, "waste_kg"] * RNG.uniform(1.3, 1.8, event_mask.sum())
).round(2)
df.loc[event_mask, "footfall"] = (
    df.loc[event_mask, "footfall"].astype(float) * RNG.uniform(1.2, 1.6, event_mask.sum())
).round().astype(int)

print(f"Special event rows: {event_mask.sum():,}")

Special event rows: 672


## 11. Waste category breakdown (organic / recyclable / landfill)

In [31]:
org_frac  = 0.52 + 0.12 * np.sin(2 * np.pi * (df["hour_frac"] - 12) / 24)
rec_frac  = 0.28 - 0.06 * np.sin(2 * np.pi * (df["hour_frac"] - 18) / 24)
land_frac = 1 - org_frac - rec_frac

w = df["waste_kg"].fillna(0).values
df["waste_organic_kg"]    = np.round(np.clip(w * org_frac,  0, None), 2)
df["waste_recyclable_kg"] = np.round(np.clip(w * rec_frac,  0, None), 2)
df["waste_landfill_kg"]   = np.round(np.clip(w * land_frac, 0, None), 2)

print("Waste categories created.")
print(df[["waste_organic_kg", "waste_recyclable_kg", "waste_landfill_kg"]].describe().round(2))

Waste categories created.
       waste_organic_kg  waste_recyclable_kg  waste_landfill_kg
count           8688.00              8688.00            8688.00
mean               0.76                 0.43               0.25
std                0.96                 0.54               0.36
min                0.00                 0.00               0.00
25%                0.07                 0.04               0.02
50%                0.16                 0.08               0.08
75%                1.29                 0.75               0.36
max                6.44                 3.36               3.13


## 12. Build target variable and select final columns

In [32]:
df["waste_kg_target"] = df["waste_kg"].clip(lower=0)
df["footfall"]        = df["footfall"].astype(int)

final_cols = [
    "timestamp", "location_id",
    "year", "month", "day", "hour", "minute",
    "day_of_week", "week_of_year", "is_weekend",
    "hour_frac", "hour_sin", "hour_cos", "month_sin", "month_cos",
    "is_holiday", "holiday_name", "special_event",
    "footfall", "footfall_raw",
    "waste_kg", "waste_kg_target",
    "waste_organic_kg", "waste_recyclable_kg", "waste_landfill_kg",
]

df_final = df[final_cols].sort_values("timestamp").reset_index(drop=True)
print(f"Final shape: {df_final.shape}")
print(df_final.dtypes)

Final shape: (8688, 25)
timestamp              datetime64[us]
location_id                       str
year                            int32
month                           int32
day                             int32
hour                            int32
minute                          int32
day_of_week                     int32
week_of_year                    int64
is_weekend                      int64
hour_frac                     float64
hour_sin                      float64
hour_cos                      float64
month_sin                     float64
month_cos                     float64
is_holiday                      int64
holiday_name                      str
special_event                     str
footfall                        int64
footfall_raw                    int64
waste_kg                      float64
waste_kg_target               float64
waste_organic_kg              float64
waste_recyclable_kg           float64
waste_landfill_kg             float64
dtype: object


## 13. Save to CSV

In [33]:
OUT = "data/synthetic_forecast_ready.csv"
df_final.to_csv(OUT, index=False)
print(f"[SUCCESS] Saved → {OUT}  ({len(df_final):,} rows × {len(df_final.columns)} cols)")

[SUCCESS] Saved → data/synthetic_forecast_ready.csv  (8,688 rows × 25 cols)


## 14. Statistical validation against original data

In [ ]:
orig_waste  = original['Waste_Weight_kg'].dropna()
synth_waste = df_final['waste_kg'].dropna()

print(f"Original  waste: mean={orig_waste.mean():.2f}, std={orig_waste.std():.2f}, "
      f"min={orig_waste.min():.2f}, max={orig_waste.max():.2f}")
print(f"Synthetic waste: mean={synth_waste.mean():.2f}, std={synth_waste.std():.2f}, "
      f"min={synth_waste.min():.2f}, max={synth_waste.max():.2f}")

print("\nBy meal (original meal column vs synthetic hour-based meal):")
meal_hour_map = {'Breakfast': (6, 10), 'Lunch': (11, 14), 'Dinner': (17, 21)}
for meal, (h_start, h_end) in meal_hour_map.items():
    orig_mean  = original[original['Meal'] == meal]['Waste_Weight_kg'].mean()
    synth_mean = df_final[
        df_final['hour'].between(h_start, h_end - 1)
    ]['waste_kg'].mean()
    print(f"  {meal}: original={orig_mean:.2f}  synthetic={synth_mean:.2f}")

[STATISTICAL MIMICKING CHECK]
Original  waste: mean=2.59, std=1.41, min=0.10, max=5.00
Synthetic waste: mean=1.44, std=1.77, min=0.00, max=10.42

By meal (original meal column vs synthetic hour-based meal):
  Breakfast: original=2.48  synthetic=2.87
  Lunch: original=2.63  synthetic=3.04
  Dinner: original=2.65  synthetic=3.00
